# Notebook 2 — OCR caractères manuscrits

In [ ]:
import sys, time
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, torch
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import Subset, random_split
from src import config, ocr_data, ocr_model, explain
sns.set_theme(style='white')
config.FIG_DIR.mkdir(parents=True, exist_ok=True)
config.RES_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(config.RANDOM_STATE)
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', DEVICE)


## Préparation

In [ ]:
train_full = ocr_data.CharIDXDataset(
    config.IMAGE_DIR / 'train-images-idx3-ubyte',
    config.IMAGE_DIR / 'train-labels-idx1-ubyte')
test_full = ocr_data.CharIDXDataset(
    config.IMAGE_DIR / 'test-images-idx3-ubyte',
    config.IMAGE_DIR / 'test-labels-idx1-ubyte')
print(f'train: {len(train_full):,}  | test: {len(test_full):,}  | classes: {train_full.num_classes}')
labels = train_full.class_labels()
print('classes:', ''.join(labels))


In [ ]:
# Q — Le dataset est-il équilibré ?
y_train_all = train_full.labels.numpy()
uniq, counts = np.unique(y_train_all, return_counts=True)
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.bar(range(len(counts)), counts, color='#1f6feb')
ax.set_xticks(range(len(counts))); ax.set_xticklabels([labels[i] for i in uniq], fontsize=8)
ax.set_title(f'Distribution des classes ({len(counts)} classes, {counts.sum():,} images)')
ax.set_ylabel('# images'); ax.axhline(counts.mean(), color='#d62728', ls='--', label=f'moy={counts.mean():.0f}')
ax.legend(); fig.tight_layout(); fig.savefig(config.FIG_DIR / 'ocr_class_balance.png', dpi=160); plt.show()
print(f'min: {counts.min()}  max: {counts.max()}  ratio max/min: {counts.max()/counts.min():.1f}x')


Dataset très déséquilibré : chiffres et majuscules ont 5-10× plus d'exemples que certaines minuscules.

In [ ]:
# Aperçu d'images (1 par classe quand possible)
fig, axes = plt.subplots(4, 16, figsize=(13, 4.2))
for i, ax in enumerate(axes.flat):
    ax.axis('off')
    if i >= train_full.num_classes:
        continue
    j = int(np.where(y_train_all == i)[0][0])
    img, lbl = train_full[j]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(labels[i], fontsize=9)
fig.suptitle('Un échantillon par classe', y=0.98)
fig.tight_layout(); fig.savefig(config.FIG_DIR / 'ocr_samples.png', dpi=160); plt.show()


## Sous-échantillonnage

20% du train (~140k images), val 10%.

In [ ]:
FRACTION = 0.2
VAL_FRACTION = 0.1
EPOCHS = 3
BATCH_SIZE = 256
rng = np.random.default_rng(config.RANDOM_STATE)
pool = rng.choice(len(train_full), size=int(len(train_full)*FRACTION), replace=False)
rng.shuffle(pool)
n_val = max(4096, int(len(pool) * VAL_FRACTION))
val_ids = pool[:n_val]; tr_ids = pool[n_val:]
train_ds = Subset(train_full, tr_ids.tolist())
val_ds = Subset(train_full, val_ids.tolist())
print(f'train_subset: {len(train_ds):,}  val_subset: {len(val_ds):,}')


## CNN

In [ ]:
model = ocr_model.CharCNN(num_classes=train_full.num_classes)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f'paramètres : {n_params:,}')


In [ ]:
t0 = time.time()
history = ocr_model.train_cnn(model, train_ds, val_ds, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=1e-3, device=DEVICE)
print(f'train time: {(time.time()-t0)/60:.1f} min')
torch.save(model.state_dict(), config.RES_DIR / 'ocr_cnn.pt')


In [ ]:
# Courbes train_loss / val_acc (over/under-fitting)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(range(1,EPOCHS+1), history['train_loss'], 'o-', color='#1f6feb')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('train loss'); axes[0].set_title('Train loss')
axes[1].plot(range(1,EPOCHS+1), history['val_acc'], 'o-', color='#2ca02c')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val accuracy'); axes[1].set_title('Validation accuracy')
fig.tight_layout(); fig.savefig(config.FIG_DIR / 'ocr_training_curves.png', dpi=160); plt.show()


## Évaluation

In [ ]:
y_true, y_pred = ocr_model.predict(model, test_full, device=DEVICE)
acc = (y_true == y_pred).mean()
print(f'TEST accuracy: {acc:.4f}  ({len(y_true):,} échantillons)')


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(train_full.num_classes))
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, cmap='magma', cbar=True, square=True, ax=ax,
            xticklabels=labels, yticklabels=labels)
ax.set_xlabel('prédit'); ax.set_ylabel('vrai'); ax.set_title('Matrice de confusion (test)')
ax.tick_params(labelsize=7)
fig.tight_layout(); fig.savefig(config.FIG_DIR / 'ocr_confusion_matrix.png', dpi=200); plt.show()


In [ ]:
# Accuracy par classe + 10 pires
per_class = (cm.diagonal() / cm.sum(axis=1).clip(min=1))
import pandas as pd
pc = pd.DataFrame({'class_idx': np.arange(len(per_class)),
                   'char': labels, 'acc': per_class.round(3),
                   'support': cm.sum(axis=1)})
pc.to_csv(config.RES_DIR / 'ocr_per_class_acc.csv', index=False)
print('10 classes les plus difficiles :')
print(pc.sort_values('acc').head(10).to_string(index=False))
print('\n10 classes les plus faciles :')
print(pc.sort_values('acc', ascending=False).head(10).to_string(index=False))


In [ ]:
# Top confusions (paires vrai → prédit)
import pandas as pd
cm0 = cm.copy(); np.fill_diagonal(cm0, 0)
ii, jj = np.unravel_index(np.argsort(cm0, axis=None)[::-1][:10], cm0.shape)
rows = [{'vrai': labels[i], 'predit': labels[j], 'count': int(cm0[i,j])} for i,j in zip(ii,jj)]
pd.DataFrame(rows)


Confusions dominantes : O/0, 1/l/I, 5/s/S. Paires intrinsèquement ambiguës en manuscrit.

## Saliency map

In [ ]:
i = 0
img, lbl = test_full[i]
char = test_full.label_to_char.get(int(lbl), '?')
sal = explain.saliency_map(model, img, target=int(lbl), device=DEVICE)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img.squeeze(), cmap='gray'); axes[0].set_title(f'image (vrai = {char})'); axes[0].axis('off')
axes[1].imshow(sal, cmap='inferno'); axes[1].set_title('|x · ∇logit|'); axes[1].axis('off')
fig.tight_layout(); fig.savefig(config.FIG_DIR / 'ocr_saliency.png', dpi=160); plt.show()
